## 1. Hugging Face Login

#### This imports the token from Kaggle Secrets, so make sure the secret is set correctly before running the code.

In [1]:
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

print("Hugging Face login successful.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login successful.


## 2. Check GPU

In [2]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

CUDA available: True
GPU count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4


## 3. Configuration

#### this is sample configuration, you are free to modify any parameter to train the best performing model

In [3]:
import os
import random
import json
import time
import math
import zipfile
import shutil

# Hugging Face dataset
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
TRAIN_FILE = "Phase_1_Pretraining/Train/train_supreme_court.json"
TEST_FILE  = "Phase_1_Pretraining/Test/test.json"

# Architecture Configuration
MODEL_DIM = 768             
MODEL_LAYERS = 12             
MODEL_HEADS = 12            
MODEL_FF = 3072               
MODEL_SEQ_LEN = 1024      
DROPOUT = 0.1
NORM_FIRST = True      

TRAIN_TOKENS = 55_000_000 
VAL_TOKENS   = 2_000_000
TRAIN_BLOCK_SIZE = 1024
EPOCHS = 1
SEED = 42
BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 4
LEARNING_RATE = 3e-4

random.seed(SEED)

TARGET_GENERATION_WORDS = 128
MIN_GENERATION_TOKENS = 180   
MAX_GENERATION_TOKENS = 320   
TEMPERATURE = 0.5             
TOP_P = 0.90                  
REPETITION_PENALTY = 1.15     

# File outputs
OUTPUT_DIR = "/kaggle/working/cs781_phase1"
MODEL_OUTPUT_DIR = os.path.join(OUTPUT_DIR, "arch2_rope_swiglu")
PREDICTION_PATH = os.path.join(OUTPUT_DIR, "prediction.json")
SUBMISSION_ZIP = "/kaggle/working/submission.zip"
LOSS_PLOT_PATH = os.path.join(OUTPUT_DIR, "training_validation_loss.png")

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Configuration loaded.(ROPE, RMSNorm SwiGLU)")

Configuration loaded.(ROPE, RMSNorm SwiGLU)


## 4. Load Tokenizer

In [4]:
import os
import json
from collections import Counter

SPECIALS = ["<pad>", "<unk>", "<s>", "</s>"]
END_OF_WORD = "</w>"

class ByteBPETokenizer:
    def __init__(self, vocab_size=16000):
        self.vocab_size = vocab_size
        self.merges = []
        self.merge_rank = {}
        self.itos = []
        self.stoi = {}

    @staticmethod
    def _word_to_symbols(word):
        byte_str = word.encode("utf-8").decode("latin-1")
        return list(byte_str) + [END_OF_WORD]

    def train(self, corpus_texts):
        word_freq = Counter()
        for text in corpus_texts:
            for word in text.split():
                word_freq[word] += 1

        word_symbols = {w: self._word_to_symbols(w) for w in word_freq}
        vocab = set(SPECIALS)
        vocab.add(END_OF_WORD)
        for i in range(256):
            vocab.add(chr(i))

        target_new_tokens = self.vocab_size - len(vocab)
        merges_done = 0

        while merges_done < target_new_tokens:
            pair_counts = Counter()
            for word, freq in word_freq.items():
                symbols = word_symbols[word]
                for i in range(len(symbols) - 1):
                    pair_counts[(symbols[i], symbols[i + 1])] += freq

            if not pair_counts:
                break
            max_count = max(pair_counts.values())
            if max_count <= 1:
                break

            best_pair = min(p for p, c in pair_counts.items() if c == max_count)
            merged_symbol = best_pair[0] + best_pair[1]
            self.merges.append(best_pair)
            self.merge_rank[best_pair] = merges_done
            vocab.add(merged_symbol)
            merges_done += 1

            for word in word_freq:
                symbols = word_symbols[word]
                new_symbols, i = [], 0
                while i < len(symbols):
                    if (i < len(symbols) - 1
                            and symbols[i] == best_pair[0]
                            and symbols[i + 1] == best_pair[1]):
                        new_symbols.append(merged_symbol)
                        i += 2
                    else:
                        new_symbols.append(symbols[i])
                        i += 1
                word_symbols[word] = new_symbols

        self.itos = (list(SPECIALS) + [END_OF_WORD]
                     + [chr(i) for i in range(256)]
                     + [p[0] + p[1] for p in self.merges])
        self.stoi = {tok: i for i, tok in enumerate(self.itos)}
        self.vocab_size = len(self.itos)
        return self

    def _apply_merges_to_word(self, word):
        symbols = self._word_to_symbols(word)
        while len(symbols) > 1:
            pairs_here = [(symbols[i], symbols[i + 1]) for i in range(len(symbols) - 1)]
            candidates = [(self.merge_rank[p], idx) for idx, p in enumerate(pairs_here) if p in self.merge_rank]
            if not candidates:
                break
            _, idx = min(candidates)
            merged = symbols[idx] + symbols[idx + 1]
            symbols = symbols[:idx] + [merged] + symbols[idx + 2:]
        return symbols

    def encode(self, text):
        ids = []
        for word in text.split():
            for sym in self._apply_merges_to_word(word):
                ids.append(self.stoi.get(sym, self.stoi["<unk>"]))
        return ids

    def decode(self, ids):
        symbols = [self.itos[i] if i < len(self.itos) else "" for i in ids]
        joined = "".join(symbols)
        words = [w for w in joined.split(END_OF_WORD) if w]
        decoded = [w.encode("latin-1").decode("utf-8", errors="replace") for w in words]
        return " ".join(decoded)

    def save(self, path):
        with open(path, "w", encoding="utf-8") as f:
            json.dump({"merges": self.merges, "itos": self.itos, "vocab_size": self.vocab_size}, f, ensure_ascii=False)

    @classmethod
    def load(cls, path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        tok = cls(vocab_size=data["vocab_size"])
        tok.itos = data["itos"]
        tok.stoi = {t: i for i, t in enumerate(tok.itos)}
        tok.merges = [tuple(p) for p in data["merges"]]
        tok.merge_rank = {p: i for i, p in enumerate(tok.merges)}
        return tok

## 5. Download Training Data From Hugging Face

In [5]:
from huggingface_hub import hf_hub_download

train_file_path = hf_hub_download(
    repo_id=HF_DATASET_REPO,
    filename=TRAIN_FILE,
    repo_type="dataset",
    token=HF_TOKEN
)

print("Training file:")
print(train_file_path)

Training file:
/root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_supreme_court.json


In [6]:
# Train on legal corpus sample or load existing
TOKENIZER_PATH = "/kaggle/input/datasets/brindaaltaccount/legal-bpe-tokenizer/custom_bpe.json"
if os.path.exists(TOKENIZER_PATH):
    print("Loading saved ByteBPETokenizer...")
    tokenizer = ByteBPETokenizer.load(TOKENIZER_PATH)
else:
    print("Training ByteBPETokenizer on legal corpus sample...")
    with open(train_file_path, "r", encoding="utf-8") as f:
        recs = json.load(f)
    sample_txts = [item["text"] for item in recs[:500] if item.get("text")]
    tokenizer = ByteBPETokenizer(vocab_size=8000)
    tokenizer.train(sample_txts)
    tokenizer.save(TOKENIZER_PATH)
    del recs, sample_txts

print(f"Custom ByteBPETokenizer ready! Vocab size: {tokenizer.vocab_size}")

Loading saved ByteBPETokenizer...
Custom ByteBPETokenizer ready! Vocab size: 8000


## 6. Create Train and Validation Tokens Set

In [7]:
with open(train_file_path,"r", encoding="utf-8") as f:
    records = json.load(f)

train_tokens = []
val_tokens = []

random.seed(SEED)

for data in records:
    text = data["text"]
    tokens = tokenizer.encode(text)

    if random.random() < 0.05:
        if len(val_tokens) < VAL_TOKENS:
            remaining = (VAL_TOKENS - len(val_tokens))
            val_tokens.extend(tokens[:remaining])
    else:
        if len(train_tokens) < TRAIN_TOKENS:
            remaining = (TRAIN_TOKENS - len(train_tokens))
            train_tokens.extend(tokens[:remaining])

    if (len(train_tokens) >= TRAIN_TOKENS and len(val_tokens) >= VAL_TOKENS):
        break

print(f"Train tokens: {len(train_tokens):,}")
print(f"Validation tokens: {len(val_tokens):,}")

Train tokens: 55,000,000
Validation tokens: 2,000,000


## 7. Create Token Blocks

In [8]:
def create_blocks(tokens, block_size):
    total_length = (len(tokens) // block_size) * block_size
    tokens = tokens[:total_length]
    return [tokens[i:i + block_size] for i in range(0, total_length, block_size)]

train_blocks = create_blocks(train_tokens, TRAIN_BLOCK_SIZE)
val_blocks = create_blocks(val_tokens, TRAIN_BLOCK_SIZE)

del train_tokens, val_tokens, records #to optimise ram

print("Training sequences:", len(train_blocks))
print("Validation sequences:", len(val_blocks))

Training sequences: 53710
Validation sequences: 1953


## 8. Create Datasets

In [9]:
from datasets import Dataset

train_dataset = Dataset.from_dict({"input_ids": train_blocks})
val_dataset = Dataset.from_dict({"input_ids": val_blocks})

print(train_dataset)
print(val_dataset)

Dataset({
    features: ['input_ids'],
    num_rows: 53710
})
Dataset({
    features: ['input_ids'],
    num_rows: 1953
})


## 9. Define The Model

In [10]:
import math
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F

# 1. Configuration
@dataclass
class Arch2Config:
    vocab_size: int
    d_model: int = 768
    n_layers: int = 12
    n_heads: int = 12
    d_ff: int = 2560
    max_seq_len: int = 1024
    dropout: float = 0.1
    rope_theta: float = 10000.0

# 2. RMSNorm (Root Mean Square Layer Normalization)
class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        variance = x.pow(2).mean(-1, keepdim=True)
        return x * torch.rsqrt(variance + self.eps) * self.weight

# 3. Rotary Positional Embeddings (RoPE)
class RotaryEmbedding(nn.Module):
    def __init__(self, dim: int, max_seq_len: int = 2048, theta: float = 10000.0):
        super().__init__()
        inv_freq = 1.0 / (theta ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)
        t = torch.arange(max_seq_len, dtype=torch.float32)
        freqs = torch.outer(t, self.inv_freq)
        emb = torch.cat((freqs, freqs), dim=-1)
        self.register_buffer("cos_cached", emb.cos(), persistent=False)
        self.register_buffer("sin_cached", emb.sin(), persistent=False)

    def forward(self, x: torch.Tensor, seq_len: int):
        return self.cos_cached[:seq_len].to(x.dtype), self.sin_cached[:seq_len].to(x.dtype)

def rotate_half(x: torch.Tensor) -> torch.Tensor:
    x1, x2 = x[..., :x.shape[-1] // 2], x[..., x.shape[-1] // 2:]
    return torch.cat((-x2, x1), dim=-1)

def apply_rotary_pos_emb(q: torch.Tensor, k: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor):
    # cos, sin: [seq_len, head_dim] -> [1, 1, seq_len, head_dim]
    cos = cos.unsqueeze(0).unsqueeze(1)
    sin = sin.unsqueeze(0).unsqueeze(1)
    q_rot = (q * cos) + (rotate_half(q) * sin)
    k_rot = (k * cos) + (rotate_half(k) * sin)
    return q_rot, k_rot

# 4. Causal Attention with RoPE
class RoPECausalAttention(nn.Module):
    def __init__(self, cfg: Arch2Config):
        super().__init__()
        assert cfg.d_model % cfg.n_heads == 0
        self.n_heads = cfg.n_heads
        self.head_dim = cfg.d_model // cfg.n_heads
        self.scale = 1.0 / math.sqrt(self.head_dim)

        self.q_proj = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.k_proj = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.v_proj = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.out_proj = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.attn_dropout = nn.Dropout(cfg.dropout)
        self.resid_dropout = nn.Dropout(cfg.dropout)

        mask = torch.tril(torch.ones(cfg.max_seq_len, cfg.max_seq_len, dtype=torch.bool))
        self.register_buffer("causal_mask", mask, persistent=False)

    def forward(self, x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        # Apply RoPE to Queries and Keys
        q, k = apply_rotary_pos_emb(q, k, cos, sin)

        scores = (q @ k.transpose(-2, -1)) * self.scale
        scores = scores.masked_fill(~self.causal_mask[:T, :T], float("-inf"))

        attn = F.softmax(scores, dim=-1)
        attn = self.attn_dropout(attn)

        z = attn @ v
        z = z.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.out_proj(z))

# 5. SwiGLU Gated Feed-Forward Network
class SwiGLUFeedForward(nn.Module):
    def __init__(self, cfg: Arch2Config):
        super().__init__()
        self.gate_proj = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)
        self.up_proj   = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)
        self.down_proj = nn.Linear(cfg.d_ff, cfg.d_model, bias=False)
        self.dropout   = nn.Dropout(cfg.dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # SwiGLU: (SiLU(x * W_gate) * (x * W_up)) * W_down
        return self.dropout(self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x)))

# 6. Transformer Block with RMSNorm
class RoPEBlock(nn.Module):
    def __init__(self, cfg: Arch2Config):
        super().__init__()
        self.norm1 = RMSNorm(cfg.d_model)
        self.attn  = RoPECausalAttention(cfg)
        self.norm2 = RMSNorm(cfg.d_model)
        self.ffn   = SwiGLUFeedForward(cfg)

    def forward(self, x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.norm1(x), cos, sin)
        x = x + self.ffn(self.norm2(x))
        return x

# 7. Complete Architecture 2 Decoder LM
class RoPESwiGLUDecoderLM(nn.Module):
    def __init__(self, cfg: Arch2Config):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.rope = RotaryEmbedding(cfg.d_model // cfg.n_heads, cfg.max_seq_len, cfg.rope_theta)
        self.blocks = nn.ModuleList([RoPEBlock(cfg) for _ in range(cfg.n_layers)])
        self.norm_f = RMSNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight  # Weight tying

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        B, T = idx.shape
        x = self.tok_emb(idx)
        cos, sin = self.rope(x, T)

        for block in self.blocks:
            x = block(x, cos, sin)

        x = self.norm_f(x)
        return self.head(x)

    @torch.inference_mode()
    def generate(self, prompt_tokens, max_new_tokens=320, min_new_tokens=180, temperature=0.5, top_p=0.9, repetition_penalty=1.15, device="cuda"):
        self.eval()
        curr = torch.tensor([prompt_tokens], dtype=torch.long, device=device)
        for step in range(max_new_tokens):
            if curr.shape[1] >= self.cfg.max_seq_len:
                break
            logits = self.forward(curr)
            next_logits = logits[0, -1, :].clone()

            # Anti-repetition penalty
            if repetition_penalty != 1.0:
                for t_id in set(curr[0].tolist()):
                    if next_logits[t_id] > 0:
                        next_logits[t_id] /= repetition_penalty
                    else:
                        next_logits[t_id] *= repetition_penalty

            next_logits = next_logits / max(temperature, 1e-5)
            sorted_logits, sorted_indices = torch.sort(next_logits, descending=True)
            cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
            sorted_indices_to_remove = cumulative_probs > top_p
            sorted_indices_to_remove[1:] = sorted_indices_to_remove[:-1].clone()
            sorted_indices_to_remove[0] = 0

            indices_to_remove = sorted_indices[sorted_indices_to_remove]
            next_logits[indices_to_remove] = float("-inf")

            probs = F.softmax(next_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            curr = torch.cat([curr, next_token.unsqueeze(0)], dim=1)

            if step >= min_new_tokens and next_token.item() in [2, 3]:
                break

        return curr[0].tolist()

device = "cuda" if torch.cuda.is_available() else "cpu"
cfg = Arch2Config(
    vocab_size=tokenizer.vocab_size,
    d_model=MODEL_DIM,
    n_layers=MODEL_LAYERS,
    n_heads=MODEL_HEADS,
    d_ff=MODEL_FF,
    max_seq_len=MODEL_SEQ_LEN
)
model = RoPESwiGLUDecoderLM(cfg).to(device)

total_params = sum(p.numel() for p in model.parameters())
print("=" * 60)
print(f"Architecture 2 Parameters: {total_params:,} ({total_params/1e6:.2f}M)")
print("=" * 60)

assert 50_000_000 <= total_params <= 300_000_000, f"Out of bounds: {total_params/1e6:.2f}M!"
print("COMPLIANCE PASS: Architecture 2 is strictly within 50M–300M parameters.")

Architecture 2 Parameters: 119,409,408 (119.41M)
COMPLIANCE PASS: Architecture 2 is strictly within 50M–300M parameters.


## 10. Training Arguments and Training

In [11]:
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

class PretrainDataset(Dataset):
    def __init__(self, blocks):
        self.blocks = [torch.tensor(b, dtype=torch.long) for b in blocks]
    def __len__(self):
        return len(self.blocks)
    def __getitem__(self, idx):
        return self.blocks[idx]

train_ds = PretrainDataset(train_blocks)
val_ds   = PretrainDataset(val_blocks)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, drop_last=True)

total_steps = (len(train_loader) // GRAD_ACCUM_STEPS) * EPOCHS
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.05, betas=(0.9, 0.95))
scheduler = CosineAnnealingLR(optimizer, T_max=total_steps, eta_min=3e-5)
scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

print(f"Starting training on {device} over {total_steps} optimizer steps...")
model.train()
step_count = 0
running_loss = 0.0
train_steps, train_loss = [], []
start_time = time.time()

for epoch in range(EPOCHS):
    for idx, batch_ids in enumerate(train_loader):
        batch_ids = batch_ids.to(device)
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            logits = model(batch_ids)
            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = batch_ids[..., 1:].contiguous()
            loss = F.cross_entropy(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1)) / GRAD_ACCUM_STEPS

        scaler.scale(loss).backward()
        running_loss += loss.item() * GRAD_ACCUM_STEPS

        if (idx + 1) % GRAD_ACCUM_STEPS == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            scheduler.step()
            step_count += 1

            if step_count % 100 == 0:
                avg_l = running_loss / 100
                train_steps.append(step_count)
                train_loss.append(avg_l)
                print(f"Step {step_count:5d}/{total_steps} | Loss: {avg_l:.4f} | Elapsed: {(time.time()-start_time)/60:.1f}m")
                running_loss = 0.0

print(f"Training completed in {(time.time()-start_time)/3600:.2f} hours.")

/tmp/ipykernel_236/519810979.py:21: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())
/tmp/ipykernel_236/519810979.py:33: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):


Starting training on cuda over 3356 optimizer steps...
Step   100/3356 | Loss: 207.8867 | Elapsed: 3.4m
Step   200/3356 | Loss: 70.5632 | Elapsed: 6.9m
Step   300/3356 | Loss: 47.2507 | Elapsed: 10.4m
Step   400/3356 | Loss: 37.2077 | Elapsed: 13.9m
Step   500/3356 | Loss: 32.7461 | Elapsed: 17.4m
Step   600/3356 | Loss: 30.5031 | Elapsed: 20.9m
Step   700/3356 | Loss: 28.8707 | Elapsed: 24.4m
Step   800/3356 | Loss: 27.6431 | Elapsed: 27.9m
Step   900/3356 | Loss: 26.6072 | Elapsed: 31.4m
Step  1000/3356 | Loss: 25.7872 | Elapsed: 34.9m
Step  1100/3356 | Loss: 24.9666 | Elapsed: 38.4m
Step  1200/3356 | Loss: 24.1004 | Elapsed: 41.9m
Step  1300/3356 | Loss: 23.4706 | Elapsed: 45.4m
Step  1400/3356 | Loss: 22.9103 | Elapsed: 48.9m
Step  1500/3356 | Loss: 22.4099 | Elapsed: 52.4m
Step  1600/3356 | Loss: 21.8301 | Elapsed: 55.9m
Step  1700/3356 | Loss: 21.4230 | Elapsed: 59.4m
Step  1800/3356 | Loss: 21.0702 | Elapsed: 62.9m
Step  1900/3356 | Loss: 20.5946 | Elapsed: 66.4m
Step  2000/3356

## 11. Perplexity

In [12]:
import os

# Create the target directory so torch can save into it
os.makedirs(MODEL_OUTPUT_DIR, exist_ok=True)

# Save the trained model weights and tokenizer for Phases 2–7
torch.save({
    "model_state_dict": model.state_dict(),
    "config": cfg,
    "vocab_size": tokenizer.vocab_size
}, os.path.join(MODEL_OUTPUT_DIR, "decoder_lm.pt"))

tokenizer.save(os.path.join(MODEL_OUTPUT_DIR, "custom_bpe.json"))

print(f"SUCCESS! Model checkpoint and tokenizer saved to: {MODEL_OUTPUT_DIR}")

SUCCESS! Model checkpoint and tokenizer saved to: /kaggle/working/cs781_phase1/arch2_rope_swiglu


In [ ]:
model.eval()
val_loss_sum = 0.0
val_count = 0

print("Evaluating validation perplexity...")
with torch.no_grad():
    for v_batch in val_loader:
        v_batch = v_batch.to(device)
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            v_logits = model(v_batch)
            v_loss = F.cross_entropy(
                v_logits[..., :-1, :].contiguous().view(-1, cfg.vocab_size),
                v_batch[..., 1:].contiguous().view(-1)
            )
        val_loss_sum += v_loss.item()
        val_count += 1

eval_loss = val_loss_sum / max(1, val_count)
perplexity = math.exp(eval_loss)
print(f"Validation Loss: {eval_loss:.4f}")
print(f"Validation Perplexity: {perplexity:.2f}")

# Save model and tokenizer for Phases 2–7
torch.save({
    "model_state_dict": model.state_dict(),
    "config": cfg,
    "vocab_size": tokenizer.vocab_size
}, os.path.join(MODEL_OUTPUT_DIR, "decoder_lm.pt"))
tokenizer.save(os.path.join(MODEL_OUTPUT_DIR, "custom_bpe.json"))
print("Model checkpoint saved to:", MODEL_OUTPUT_DIR)

## 13. Plot Training and Validation Loss

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))

# Plot training loss curve
if train_steps:
    plt.plot(train_steps, train_loss, label="Training Loss", color="#1f77b4", linewidth=2)

# Plot validation loss evaluated at the end
if 'eval_loss' in locals() and train_steps:
    final_step = train_steps[-1]
    plt.scatter([final_step], [eval_loss], color="red", s=100, zorder=5, 
                label=f"Final Val Loss ({eval_loss:.2f})")
    plt.axhline(y=eval_loss, color="red", linestyle="--", alpha=0.5, 
                label=f"Val Loss Baseline ({eval_loss:.2f})")

plt.xlabel("Optimizer Steps", fontsize=12)
plt.ylabel("Cross-Entropy Loss", fontsize=12)
plt.title("Phase 1: Training & Validation Loss (Assignment 3 DecoderLM)", fontsize=13)
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend(fontsize=11)
plt.tight_layout()
plt.savefig(LOSS_PLOT_PATH, dpi=200)
plt.show()

print("Loss plot saved to:", LOSS_PLOT_PATH)

## 14. Save Final Model to Kaggle Output

In [ ]:
# # trainer.save_model(MODEL_OUTPUT_DIR)
# tokenizer.save_pretrained(MODEL_OUTPUT_DIR)

# print("Final model saved to:")
# print(MODEL_OUTPUT_DIR)

## 15. Push Model to Hugging Face Hub

In [ ]:
model.push_to_hub(HF_MODEL_REPO, token=HF_TOKEN)
tokenizer.push_to_hub(HF_MODEL_REPO, token=HF_TOKEN)

# print("Model and tokenizer pushed successfully.")

## 16. Load Test Data from HuggingFace

In [13]:
test_file_path = hf_hub_download(
    repo_id=HF_DATASET_REPO,
    filename=TEST_FILE,
    repo_type="dataset",
    token=HF_TOKEN
)

print("Test file:", test_file_path)

test.json: 0.00B [00:00, ?B/s]

Test file: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Test/test.json


## 17. Load Test Prompts

In [14]:
with open(test_file_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)

print("Number of test examples:", len(test_data))

Number of test examples: 1000


## 18. Generate 128 Words
#### (words, not tokens)

In [15]:
def generate_128_words(prompt_text, model, tokenizer, device, target_words=128):
    """
    High-speed 128-word autoregressive generation.
    Stops as soon as 128 words are produced.
    """
    model.eval()
    
    # 1. Encode prompt and keep only the last 256 tokens for fast attention computation
    input_ids = tokenizer.encode(prompt_text)
    if len(input_ids) > 256:
        input_ids = input_ids[-256:]
        
    generated_ids = []
    curr_ids = list(input_ids)
    
    # Pre-allocate input tensor
    x = torch.tensor([curr_ids], dtype=torch.long, device=device)
    
    with torch.no_grad():
        # 150 tokens is plenty to reach 128 words
        for step in range(160):
            # Keep context window under 384 tokens for lightning-fast forward pass
            context = x if x.size(1) <= 384 else x[:, -384:]
            
            logits = model(context)
            next_logits = logits[0, -1, :] / 0.7  # Temperature = 0.7
            
            # Fast repetition penalty on recent 20 generated tokens
            if generated_ids:
                recent_tokens = set(generated_ids[-20:])
                for tok in recent_tokens:
                    next_logits[tok] /= 1.15
            
            # Top-p (nucleus) sampling (top 90%)
            sorted_logits, sorted_indices = torch.sort(next_logits, descending=True)
            cumulative_probs = torch.cumsum(torch.softmax(sorted_logits, dim=-1), dim=-1)
            sorted_indices_to_remove = cumulative_probs > 0.90
            sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
            sorted_indices_to_remove[..., 0] = 0
            indices_to_remove = sorted_indices[sorted_indices_to_remove]
            next_logits[indices_to_remove] = -float('Inf')
            
            probs = torch.softmax(next_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1).item()
            
            generated_ids.append(next_token)
            
            # Append token to input tensor
            next_tensor = torch.tensor([[next_token]], dtype=torch.long, device=device)
            x = torch.cat([x, next_tensor], dim=1)
            
            # Check word count every 10 tokens to break early
            if step >= 110 and step % 10 == 0:
                gen_text = tokenizer.decode(generated_ids)
                if len(gen_text.strip().split()) >= target_words:
                    break
                    
    # 2. Decode text continuation
    full_gen_text = tokenizer.decode(generated_ids).strip()
    words = full_gen_text.split()
    
    if len(words) >= target_words:
        return " ".join(words[:target_words])
    
    # In the rare case of fewer words, repeat until strictly target_words
    if not words:
        words = ["the", "court", "held", "that"]
    while len(words) < target_words:
        words.extend(words[:target_words - len(words)])
        
    return " ".join(words[:target_words])

## 20. Generate Predictions for the Test Set

In [16]:
import os
import json

# Auto-resume: load existing predictions so you NEVER lose progress if session restarts
predictions = {}
if os.path.exists(PREDICTION_PATH):
    try:
        with open(PREDICTION_PATH, "r", encoding="utf-8") as f:
            predictions = json.load(f)
        print(f"Resuming! Found {len(predictions)} already completed predictions.")
    except Exception:
        predictions = {}

items = list(test_data.items())
total = len(items)

# Filter out already generated samples
remaining_items = [(ex_id, rec) for ex_id, rec in items if str(ex_id) not in predictions]
print(f"Total: {total} | Already done: {len(predictions)} | Remaining: {len(remaining_items)}")

model.eval()

for idx, (ex_id, rec) in enumerate(remaining_items):
    # Generates with target words
    continuation = generate_128_words(rec["text"], model, tokenizer, device, target_words=TARGET_GENERATION_WORDS)
    predictions[str(ex_id)] = {"text": continuation}
    
    # Auto-save to disk every 25 samples so you never lose progress!
    if (idx + 1) % 25 == 0 or (idx + 1) == len(remaining_items):
        with open(PREDICTION_PATH, "w", encoding="utf-8") as f:
            json.dump(predictions, f, indent=2, ensure_ascii=False)
        print(f"Saved progress: {len(predictions)}/{total} ({len(predictions)/total*100:.1f}%)")

print("Inference completed! All samples generated and saved.")

Total: 1000 | Already done: 0 | Remaining: 1000
Saved progress: 25/1000 (2.5%)
Saved progress: 50/1000 (5.0%)
Saved progress: 75/1000 (7.5%)
Saved progress: 100/1000 (10.0%)
Saved progress: 125/1000 (12.5%)
Saved progress: 150/1000 (15.0%)
Saved progress: 175/1000 (17.5%)
Saved progress: 200/1000 (20.0%)
Saved progress: 225/1000 (22.5%)
Saved progress: 250/1000 (25.0%)
Saved progress: 275/1000 (27.5%)
Saved progress: 300/1000 (30.0%)
Saved progress: 325/1000 (32.5%)
Saved progress: 350/1000 (35.0%)
Saved progress: 375/1000 (37.5%)
Saved progress: 400/1000 (40.0%)
Saved progress: 425/1000 (42.5%)
Saved progress: 450/1000 (45.0%)
Saved progress: 475/1000 (47.5%)
Saved progress: 500/1000 (50.0%)
Saved progress: 525/1000 (52.5%)
Saved progress: 550/1000 (55.0%)
Saved progress: 575/1000 (57.5%)
Saved progress: 600/1000 (60.0%)
Saved progress: 625/1000 (62.5%)
Saved progress: 650/1000 (65.0%)
Saved progress: 675/1000 (67.5%)
Saved progress: 700/1000 (70.0%)
Saved progress: 725/1000 (72.5%)
S

## 21. Save prediction.json

In [17]:
with open(PREDICTION_PATH, "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)

if os.path.exists(SUBMISSION_ZIP):
    os.remove(SUBMISSION_ZIP)
with zipfile.ZipFile(SUBMISSION_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(PREDICTION_PATH, arcname="prediction.json")

## 22. Create CodaBench Submission ZIP

In [ ]:
with zipfile.ZipFile(SUBMISSION_ZIP, "r") as zipf:
    contents = zipf.namelist()
    print("Archive contents:", contents)
    assert contents == ["prediction.json"], "ERROR: Archive must contain ONLY prediction.json at root!"
print("\nSUCCESS! /kaggle/working/submission.zip is ready for Codabench submission.")